In [54]:
# ============================================================
# ⚙️  CONFIGURAZIONE MISCELE  — modifica solo questo blocco
# ============================================================
# Per ogni miscela definisci:
#   name            → etichetta (corrisponde anche alla cartella output/<name>)
#   input_dir       → cartella dei file di misura
#   src_file        → file con sorgente
#   nosrc_file      → file senza sorgente
#   src_voltage_tag / src_current_tag    → canali nel file con sorgente
#   nosrc_voltage_tag / nosrc_current_tag→ canali nel file senza sorgente
#   gain_peak_min_voltage                → soglia minima V per i punti di guadagno (None = nessun taglio)

MIXTURE_CONFIGS = [
    {
        'name':              'Ar_CO2_Iso_93_5_2',
        'input_dir':         'Ar_CO2_Iso_93_5_2',
        'src_file':          'MM2DNAP_Cor_Sorg_5_2.txt',
        'nosrc_file':        'MM2DNAP_Cor_No_Sorg_5_2.txt',
        'src_voltage_tag':   'V2',
        'src_current_tag':   'I2',
        'nosrc_voltage_tag': 'V2',
        'nosrc_current_tag': 'I2',
        'gain_peak_min_voltage': None,
    },
    {
        'name':              'Ar_CO2_Iso_88_10_2',
        'input_dir':         'Ar_CO2_Iso_88_10_2',
        'src_file':          'Cor_MM2DNAP_Sorg_10_2.txt',
        'nosrc_file':        'Cor_MM2DNAP_No_Sorg_10_2.txt',
        'src_voltage_tag':   'V2',
        'src_current_tag':   'I2',
        'nosrc_voltage_tag': 'V2',
        'nosrc_current_tag': 'I2',
        'gain_peak_min_voltage': 530.0,
    },
]


# Plot V2 e I2 per miscela
Questo notebook legge due file di testo (con sorgente e senza sorgente), estrae `V2` e `I2`, costruisce anche la curva `I2_diff = I2_sorg - I2_no_sorg` (interpolata in funzione della tensione) e disegna tutto su canvas ROOT.

La logica è organizzata per **miscele** (non per camere).
Esegui le celle in ordine e poi le celle finali per salvare canvas e tabelle in `output/<miscela>/`.

In [41]:
import re
from pathlib import Path
import pandas as pd

In [42]:
def parse_file(path, voltage_tag='V2', current_tag='I2'):
    """Legge un file di testo e restituisce DataFrame standard con colonne: time, V, I.
    voltage_tag/current_tag indicano i canali da leggere dall'header del file.
    """
    p = Path(path)
    text = p.read_text(encoding='utf-8', errors='ignore').splitlines()

    header_idx = None
    header_tokens = None
    for i, line in enumerate(text):
        if voltage_tag in line and current_tag in line:
            header_tokens = [tok.strip() for tok in line.split('\t') if tok.strip()]
            header_idx = i
            break

    if header_idx is None:
        raise ValueError(f"Header con '{voltage_tag}' e '{current_tag}' non trovato in {p}")

    df = pd.read_csv(p, sep='\t', header=None, skiprows=header_idx + 1, engine='python')

    data_tokens = [tok for tok in header_tokens if tok != '|']
    v_token_idx = next((ii for ii, t in enumerate(data_tokens) if re.search(rf'\b{re.escape(voltage_tag)}\b', t)), None)
    i_token_idx = next((ii for ii, t in enumerate(data_tokens) if re.search(rf'\b{re.escape(current_tag)}\b', t)), None)

    if v_token_idx is None or i_token_idx is None:
        raise ValueError(f"Non riesco a determinare gli indici di {voltage_tag} o {current_tag} dall'header")

    if v_token_idx >= len(df.columns) or i_token_idx >= len(df.columns):
        raise ValueError(
            f"Indici {voltage_tag} ({v_token_idx}) o {current_tag} ({i_token_idx}) fuori dai limiti. "
            f"DataFrame ha {len(df.columns)} colonne."
        )

    time_col_idx = 1 if len(df.columns) > 1 else 0
    timestamp_series = df.iloc[:, time_col_idx].astype(str)

    times = pd.to_datetime(timestamp_series, dayfirst=True, errors='coerce')
    if times.isna().all():
        times = pd.to_datetime(timestamp_series, errors='coerce')

    v_series = pd.to_numeric(df.iloc[:, v_token_idx], errors='coerce')
    i_series = pd.to_numeric(df.iloc[:, i_token_idx], errors='coerce')

    out = pd.DataFrame({'time': times, 'V': v_series, 'I': i_series})
    out = out.dropna(subset=['time'])
    return out

In [43]:
def plot_series(dfs, labels=None, savepath=None, canvas_name=None):
    import array
    import ROOT
    import numpy as np

    ROOT.gROOT.SetBatch(True)
    ROOT.gStyle.SetOptStat(0)
    if canvas_name is None:
        canvas_name = f"c_{ROOT.TUUID().AsString().replace('-', '_')}"
    c = ROOT.TCanvas(canvas_name, 'Voltage and Current', 1200, 700)
    c.SetGrid()
    c.SetLeftMargin(0.12)
    c.SetRightMargin(0.18)

    leg = ROOT.TLegend(0.14, 0.74, 0.38, 0.89)
    leg.SetFillStyle(0)
    leg.SetBorderSize(0)
    leg.SetTextSize(0.03)

    i_colors = [
        ROOT.TColor.GetColor('#D55E00'),
        ROOT.TColor.GetColor('#009E73'),
        ROOT.TColor.GetColor('#CC79A7'),
    ]

    all_times = [pd.to_datetime(df['time']) for df in dfs if 'time' in df.columns and not df['time'].isna().all()]
    all_v = np.concatenate([df['V'].astype(float).to_numpy() for df in dfs if 'V' in df.columns and not df['V'].isna().all()])
    all_i = np.concatenate([df['I'].astype(float).to_numpy() for df in dfs if 'I' in df.columns and not df['I'].isna().all()])

    vmin, vmax = np.nanmin(all_v), np.nanmax(all_v)
    imin, imax = np.nanmin(all_i), np.nanmax(all_i)
    if (imax - imin) == 0:
        scale = 1.0
    else:
        scale = (vmax - vmin) / (imax - imin)
    offset = vmin - imin * scale

    first = True
    voltage_in_legend = False
    drawn_objects = []
    last_graph = None
    t0 = min(ts.min() for ts in all_times) if all_times else pd.Timestamp.now()

    for i, df in enumerate(dfs):
        lab = labels[i] if labels else f'I {i+1}'

        x_vals = (pd.to_datetime(df['time']) - t0).dt.total_seconds().astype(float).to_numpy()

        if 'V' in df.columns and not df['V'].isna().all():
            y_vals_v = df['V'].astype(float).to_numpy()
            g_v = ROOT.TGraph(len(x_vals), array.array('d', x_vals), array.array('d', y_vals_v))
            g_v.SetLineColor(ROOT.kBlue)
            g_v.SetLineWidth(2)
            g_v.SetMarkerStyle(0)
            g_v.SetMarkerSize(0)
            g_v.SetTitle('')
            g_v.GetXaxis().SetTitle('Tempo trascorso [s]')
            g_v.GetYaxis().SetTitle('Tensione [V]')
            g_v.GetYaxis().SetTitleColor(ROOT.kBlue + 1)
            g_v.GetYaxis().SetLabelColor(ROOT.kBlue + 1)
            g_v.GetYaxis().SetAxisColor(ROOT.kBlue + 1)

            if first:
                g_v.Draw('AL')
                first = False
            else:
                g_v.Draw('L same')
            if not voltage_in_legend:
                leg.AddEntry(g_v, 'V2', 'l')
                voltage_in_legend = True
            drawn_objects.append(g_v)
            last_graph = g_v

        if 'I' in df.columns and not df['I'].isna().all():
            y_vals_i = df['I'].astype(float).to_numpy()
            y_vals_i_scaled = y_vals_i * scale + offset
            g_i = ROOT.TGraph(len(x_vals), array.array('d', x_vals), array.array('d', y_vals_i_scaled))
            color = i_colors[i % len(i_colors)]
            g_i.SetLineColor(color)
            g_i.SetLineWidth(2)
            g_i.SetMarkerStyle(0)
            g_i.SetMarkerSize(0)
            g_i.Draw('L same')
            leg.AddEntry(g_i, lab, 'l')
            drawn_objects.append(g_i)

    # Aggiorna il pad prima di leggere i limiti visualizzati
    c.Modified()
    c.Update()

    if last_graph is not None:
        # Asse destro ancorato al bordo destro reale del pad
        x_right = ROOT.gPad.GetUxmax()
        y_low = ROOT.gPad.GetUymin()
        y_high = ROOT.gPad.GetUymax()
        axis = ROOT.TGaxis(x_right, y_low, x_right, y_high, imin, imax, 510, '+R')
        axis.SetTitle('Corrente [uA]')
        axis.SetTitleColor(ROOT.kRed)
        axis.SetLabelColor(ROOT.kRed)
        axis.SetLineColor(ROOT.kRed)
        axis.Draw()
        drawn_objects.append(axis)

    leg.Draw()
    drawn_objects.append(leg)
    c.Modified()
    c._keepalive = drawn_objects

    if savepath:
        root_path = f'{savepath}.root'
        png_path = f'{savepath}.png'
        c.SaveAs(root_path)
        c.SaveAs(png_path)
        print(f'Saved ROOT canvas to {root_path}')
        print(f'Saved PNG to {png_path}')
    return c

In [44]:
# Cella legacy (versione a camere) disattivata
print("Questa cella legacy a camere è stata disattivata.")
print("Usa la pipeline per miscele (cella 8) e il plot per miscele (cella 6).")

Questa cella legacy a camere è stata disattivata.
Usa la pipeline per miscele (cella 8) e il plot per miscele (cella 6).


In [52]:
# Plot separato di I2_diff vs V2 per tutte le miscele
import array
from pathlib import Path

import numpy as np
import pandas as pd
import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)

output_dir = Path('output')
plot_summary = []
VOLTAGE_BIN_STEP = 5.0

mixture_configs = globals().get('MIXTURE_CONFIGS', [])
mixtures = [cfg.get('name') for cfg in mixture_configs if isinstance(cfg, dict) and cfg.get('name')]

if not mixtures:
    mixtures = [p.name for p in output_dir.iterdir() if p.is_dir()] if output_dir.exists() else []

for mixture in mixtures:
    mixture_out = output_dir / mixture
    csv_candidates = [
        mixture_out / f'{mixture}_I_table.csv',
        mixture_out / f'{mixture}_I_table_debug.csv',
    ]
    csv_path = next((p for p in csv_candidates if p.exists()), None)
    if csv_path is None:
        print(f'[{mixture}] nessun CSV trovato in {mixture_out}')
        continue

    table_df = pd.read_csv(csv_path)

    v_candidates = [c for c in table_df.columns if c.startswith('V2')]
    if not v_candidates:
        v_candidates = [c for c in table_df.columns if c.startswith('V_')]
    if not v_candidates:
        raise KeyError(f'[{mixture}] non trovo la colonna di tensione nel CSV {csv_path}')
    v_col = v_candidates[0]

    diff_col = 'I2_diff' if 'I2_diff' in table_df.columns else ('I_diff' if 'I_diff' in table_df.columns else 'I3_diff')
    if diff_col not in table_df.columns:
        raise KeyError(f'[{mixture}] non trovo la colonna differenza nel CSV {csv_path}')

    work = table_df[[v_col, diff_col]].copy()
    work.columns = ['V', 'I_diff']
    # Binning a step di 5 V: es. 574.9 e 575.1 finiscono entrambi a 575
    work['V_round'] = (work['V'] / VOLTAGE_BIN_STEP).round() * VOLTAGE_BIN_STEP

    grouped = work.groupby('V_round')['I_diff'].agg(['mean', 'std', 'count']).reset_index()
    grouped.columns = ['V', 'I_diff_mean', 'I_diff_std', 'n']
    grouped = grouped[grouped['I_diff_mean'] > 0].sort_values('V').reset_index(drop=True)

    if grouped.empty:
        print(f'[{mixture}] nessun punto positivo disponibile, salto il plot.')
        continue

    print(f'\n===== {mixture} =====')
    print(f'CSV usato: {csv_path}')
    print(f'Binning tensione: step = {VOLTAGE_BIN_STEP:g} V')
    print('Corrente media (I2_diff) per step di tensione:')
    print(grouped.to_string(index=False))

    v_arr = array.array('d', grouped['V'].to_numpy(dtype=float))
    i_arr = array.array('d', grouped['I_diff_mean'].to_numpy(dtype=float))
    ev_arr = array.array('d', np.zeros(len(grouped), dtype=float))
    ei_arr = array.array('d', (grouped['I_diff_std'] / np.sqrt(grouped['n'])).fillna(0).to_numpy(dtype=float))

    g = ROOT.TGraphErrors(len(v_arr), v_arr, i_arr, ev_arr, ei_arr)
    g.SetTitle('')
    g.SetMarkerStyle(20)
    g.SetMarkerSize(0.8)
    g.SetMarkerColor(ROOT.TColor.GetColor('#D55E00'))
    g.SetLineColor(ROOT.TColor.GetColor('#D55E00'))

    fit_available = len(grouped) >= 3
    fit_overlay = None
    a_val = None
    b_val = None
    n_fit = 0

    if fit_available:
        n_fit = max(2, len(grouped) // 2)
        n_fit = min(n_fit, len(grouped) - 1)
        fit_slice = slice(1, 1 + n_fit)

        log_i = np.log(grouped['I_diff_mean'].iloc[fit_slice].to_numpy(dtype=float))
        v_fit = grouped['V'].iloc[fit_slice].to_numpy(dtype=float)
        g_fit_tmp = ROOT.TGraph(n_fit, array.array('d', v_fit), array.array('d', log_i))
        fit_result = g_fit_tmp.Fit('pol1', 'SQ')
        a_val = fit_result.Value(0)
        b_val = fit_result.Value(1)
        print(f'Fit log(I) = a + b*V  =>  a={a_val:.4f}, b={b_val:.6f}')

        fit_overlay = ROOT.TF1(
            f'fit_overlay_{mixture}',
            f'exp({a_val}) * exp({b_val}*x)',
            float(grouped['V'].iloc[1]),
            float(grouped['V'].iloc[1 + n_fit - 1])
        )
        fit_overlay.SetLineColor(ROOT.kBlue)
        fit_overlay.SetLineWidth(2)
        fit_overlay.SetLineStyle(2)
    else:
        print(f'[{mixture}] troppo pochi punti per un fit lineare dopo aver escluso il primo: fit saltato.')

    c2 = ROOT.TCanvas(f'c_{mixture}_Idiff_vs_V', f'I_diff vs V - {mixture}', 900, 800)
    c2.Divide(1, 2)

    pad1 = c2.cd(1)
    pad1.SetLogy()
    pad1.SetGrid()
    pad1.SetBottomMargin(0.02)
    pad1.SetTopMargin(0.08)

    g.Draw('AP')
    g.GetXaxis().SetTitle('Tensione [V]')
    g.GetYaxis().SetTitle('I_{2,diff} media [#muA]')
    g.GetXaxis().SetLabelSize(0)
    g.GetXaxis().SetTitleSize(0)

    leg2 = ROOT.TLegend(0.15, 0.75, 0.58, 0.90)
    leg2.SetFillStyle(0)
    leg2.SetBorderSize(0)
    leg2.SetTextSize(0.05)
    leg2.AddEntry(g, 'I_{2,diff} = I_{2,sorg} - I_{2,no sorg}', 'p')
    if fit_overlay is not None:
        fit_overlay.Draw('same')
        leg2.AddEntry(fit_overlay, f'Fit exp: a={a_val:.3f}, b={b_val:.5f}', 'l')
    leg2.Draw()

    pad2 = c2.cd(2)
    pad2.SetGrid()
    pad2.SetTopMargin(0.02)
    pad2.SetBottomMargin(0.18)

    if fit_overlay is not None:
        ratio_y = []
        ratio_ey = []
        for _, row in grouped.iterrows():
            fit_val = np.exp(a_val) * np.exp(b_val * row['V'])
            if fit_val != 0:
                ratio_y.append(row['I_diff_mean'] / fit_val)
                n = row['n'] if row['n'] > 0 else 1
                ratio_ey.append((row['I_diff_std'] / np.sqrt(n)) / fit_val)
            else:
                ratio_y.append(np.nan)
                ratio_ey.append(0)

        ratio_y = np.array(ratio_y, dtype=float)
        ratio_ey = np.array(ratio_ey, dtype=float)
        valid = np.isfinite(ratio_y)
        v_valid = np.array(v_arr, dtype=float)[valid]
        zero_valid = np.array(ev_arr, dtype=float)[valid]

        g_ratio = ROOT.TGraphErrors(
            int(valid.sum()),
            array.array('d', v_valid),
            array.array('d', ratio_y[valid]),
            array.array('d', zero_valid),
            array.array('d', ratio_ey[valid]),
        )
        g_ratio.SetTitle('')
        g_ratio.SetMarkerStyle(20)
        g_ratio.SetMarkerSize(0.8)
        g_ratio.SetMarkerColor(ROOT.TColor.GetColor('#009E73'))
        g_ratio.SetLineColor(ROOT.TColor.GetColor('#009E73'))
        g_ratio.Draw('AP')
        g_ratio.GetXaxis().SetTitle('Tensione [V]')
        g_ratio.GetYaxis().SetTitle('I_{2,diff} / Fit')

        line1 = ROOT.TLine(float(grouped['V'].min()), 1.0, float(grouped['V'].max()), 1.0)
        line1.SetLineColor(ROOT.kBlue)
        line1.SetLineStyle(2)
        line1.SetLineWidth(2)
        line1.Draw()

    c2.Modified()
    c2.Update()

    out_base = mixture_out / f'{mixture}_Idiff_vs_V'
    c2.SaveAs(str(out_base.with_suffix('.root')))
    c2.SaveAs(str(out_base.with_suffix('.png')))
    print(f'Salvato: {out_base}.root / .png')

    plot_summary.append({
        'miscela': mixture,
        'csv': str(csv_path),
        'plot_root': str(out_base.with_suffix('.root')),
        'plot_png': str(out_base.with_suffix('.png')),
        'n_points': int(len(grouped)),
        'fit_used': bool(fit_available)
    })

print('\n===== RIEPILOGO PLOT I_diff vs V =====')
print(pd.DataFrame(plot_summary).to_string(index=False) if plot_summary else 'Nessun plot generato.')


===== Ar_CO2_Iso_93_5_2 =====
CSV usato: output/Ar_CO2_Iso_93_5_2/Ar_CO2_Iso_93_5_2_I_table.csv
Binning tensione: step = 5 V
Corrente media (I2_diff) per step di tensione:
    V  I_diff_mean  I_diff_std   n
490.0     0.001245    0.000128 320
500.0     0.001652    0.000122 160
510.0     0.002164    0.000099 280
515.0     0.002505    0.000096 290
520.0     0.002974    0.000125 410
525.0     0.003512    0.000150 340
530.0     0.004305    0.000191 410
535.0     0.005452    0.000205 420
540.0     0.007475    0.000875 330
545.0     0.011912    0.001071 480
550.0     0.023336    0.003966 560
555.0     0.046982    0.004868 560
560.0     0.099956    0.011254 630
565.0     0.183833    0.024812 900
570.0     0.249711    0.045018 490
575.0     0.296274    0.034884 502
Fit log(I) = a + b*V  =>  a=-24.9449, b=0.036887
Salvato: output/Ar_CO2_Iso_93_5_2/Ar_CO2_Iso_93_5_2_Idiff_vs_V.root / .png

===== Ar_CO2_Iso_88_10_2 =====
CSV usato: output/Ar_CO2_Iso_88_10_2/Ar_CO2_Iso_88_10_2_I_table.csv
Binning 

Info in <TCanvas::SaveAs>: ROOT file output/Ar_CO2_Iso_93_5_2/Ar_CO2_Iso_93_5_2_Idiff_vs_V.root has been created
Info in <TCanvas::Print>: png file output/Ar_CO2_Iso_93_5_2/Ar_CO2_Iso_93_5_2_Idiff_vs_V.png has been created
Info in <TCanvas::SaveAs>: ROOT file output/Ar_CO2_Iso_88_10_2/Ar_CO2_Iso_88_10_2_Idiff_vs_V.root has been created
Info in <TCanvas::Print>: png file output/Ar_CO2_Iso_88_10_2/Ar_CO2_Iso_88_10_2_Idiff_vs_V.png has been created


### Note
- Se i tuoi file hanno formati diversi per il timestamp o separatori differenti, copia la cella `parse_file` e adattala.
- Per plottare più file insieme, imposta `dfs = [parse_file(p1), parse_file(p2), ...]` e chiama `plot_series(dfs, labels=[...])`.
- Se vuoi che converta la corrente da uA ad A, posso aggiungere un'opzione per la conversione.

In [51]:
# Pipeline per miscele: lettura con/senza sorgente su V2/I2
from pathlib import Path
import numpy as np
import pandas as pd
import ROOT

output_dir = Path('output')
output_dir.mkdir(parents=True, exist_ok=True)


def interpolate_current_by_voltage(v_ref, v_curve, i_curve):
    """Interpolazione lineare di I(V) su v_ref. Fuori range restituisce NaN."""
    ref = pd.to_numeric(v_ref, errors='coerce').to_numpy(dtype=float)

    tmp = pd.DataFrame({'V': pd.to_numeric(v_curve, errors='coerce'), 'I': pd.to_numeric(i_curve, errors='coerce')})
    tmp = tmp.dropna(subset=['V', 'I'])
    if tmp.empty:
        return np.full(ref.shape, np.nan, dtype=float)

    grouped = tmp.groupby('V', as_index=False)['I'].mean().sort_values('V')
    v_sorted = grouped['V'].to_numpy(dtype=float)
    i_sorted = grouped['I'].to_numpy(dtype=float)

    if len(v_sorted) == 1:
        out = np.full(ref.shape, np.nan, dtype=float)
        out[np.isfinite(ref) & np.isclose(ref, v_sorted[0])] = i_sorted[0]
        return out

    return np.interp(ref, v_sorted, i_sorted, left=np.nan, right=np.nan)


summary_rows = []

for cfg in MIXTURE_CONFIGS:
    mixture     = cfg['name']
    input_dir   = Path(cfg['input_dir'])
    src_v_tag   = cfg['src_voltage_tag']
    src_i_tag   = cfg['src_current_tag']
    nosrc_v_tag = cfg['nosrc_voltage_tag']
    nosrc_i_tag = cfg['nosrc_current_tag']

    src_path   = input_dir / cfg['src_file']
    nosrc_path = input_dir / cfg['nosrc_file']

    if not src_path.exists() or not nosrc_path.exists():
        print(f"[{mixture}] file mancanti: {src_path} / {nosrc_path} — salto.")
        continue

    src_df   = parse_file(src_path,   voltage_tag=src_v_tag,   current_tag=src_i_tag).sort_values('time')
    nosrc_df = parse_file(nosrc_path, voltage_tag=nosrc_v_tag, current_tag=nosrc_i_tag).sort_values('time')

    src_selected_df = src_df[['time', 'V', 'I']].rename(
        columns={'V': f'V_{src_v_tag}_sorg', 'I': f'I_{src_i_tag}_sorg'}
    )
    nosrc_selected_df = nosrc_df[['time', 'V', 'I']].rename(
        columns={'V': f'V_{nosrc_v_tag}_no_sorg', 'I': f'I_{nosrc_i_tag}_no_sorg'}
    )

    i_no_sorg_interp = interpolate_current_by_voltage(
        src_df['V'], nosrc_df['V'], nosrc_df['I'],
    )

    merged = src_df[['time', 'V', 'I']].copy()
    merged.columns = ['time', 'V2', 'I2_sorg']
    merged['I2_no_sorg_interp'] = i_no_sorg_interp
    merged['I2_diff'] = merged['I2_sorg'] - merged['I2_no_sorg_interp']

    # Regola richiesta: se manca I con sorgente o I senza sorgente (interp), il punto viene escluso del tutto
    valid_curr_mask = merged['I2_sorg'].notna() & merged['I2_no_sorg_interp'].notna()
    n_total = len(merged)
    n_valid = int(valid_curr_mask.sum())
    n_dropped_missing_current = int((~valid_curr_mask).sum())

    merged_valid = merged.loc[valid_curr_mask].copy().reset_index(drop=True)

    sorg_df  = merged_valid[['time', 'V2', 'I2_sorg']].rename(columns={'V2': 'V', 'I2_sorg': 'I'}).copy()
    nosorg_df = merged_valid[['time', 'V2', 'I2_no_sorg_interp']].rename(columns={'V2': 'V', 'I2_no_sorg_interp': 'I'}).copy()
    diff_df  = merged_valid[['time', 'V2', 'I2_diff']].rename(columns={'V2': 'V', 'I2_diff': 'I'}).copy()

    if merged_valid.empty:
        print(f"[{mixture}] nessun punto valido dopo il filtro correnti (I_sorg / I_no_sorg). Salto output.")
        summary_rows.append({
            'miscela':                    mixture,
            'canale_tensione_sorg':        src_v_tag,
            'canale_corrente_sorg':        src_i_tag,
            'canale_tensione_no_sorg':     nosrc_v_tag,
            'canale_corrente_no_sorg':     nosrc_i_tag,
            'n_righe_originali':           int(n_total),
            'n_righe':                     0,
            'n_scartate_corrente_mancante': int(n_dropped_missing_current),
            'n_interp_valide':             0,
            'n_fuori_range_tensione':      int(n_dropped_missing_current),
            'I2_no_sorg_abs_max':          float('nan'),
            'root':                        '',
            'png':                         '',
            'csv':                         '',
            'csv_input_sorg':              '',
            'csv_input_no_sorg':           '',
        })
        continue

    mixture_out = output_dir / mixture
    mixture_out.mkdir(parents=True, exist_ok=True)

    output_base = mixture_out / f'{mixture}_combined'
    print(f'[{mixture}] output_base = {output_base}')

    canvas = plot_series(
        [sorg_df, nosorg_df, diff_df],
        labels=[f'{mixture} I2_sorg', f'{mixture} I2_no_sorg_interp(V2)', f'{mixture} I2_diff'],
        savepath=None,
        canvas_name=f'c_{mixture}',
    )

    canvas.SetName(f'c_{mixture}')
    root_path = output_base.with_suffix('.root')
    root_file = ROOT.TFile(str(root_path), 'RECREATE')
    root_file.cd()
    canvas.Write(canvas.GetName())
    root_file.Close()

    png_path = output_base.with_suffix('.png')
    canvas.SaveAs(str(png_path))

    table_df  = merged_valid[['time', 'V2', 'I2_sorg', 'I2_no_sorg_interp', 'I2_diff']].copy()
    table_df  = table_df.rename(columns={'V2': 'V2_ref'})
    csv_path  = mixture_out / f'{mixture}_I_table.csv'
    table_df.to_csv(csv_path, index=False)

    src_cols_csv   = mixture_out / f'{mixture}_selected_columns_sorg.csv'
    nosrc_cols_csv = mixture_out / f'{mixture}_selected_columns_no_sorg.csv'
    src_selected_df.to_csv(src_cols_csv, index=False)
    nosrc_selected_df.to_csv(nosrc_cols_csv, index=False)

    print(f"\n=== {mixture} ===")
    print(f"Colonne: sorgente=({src_v_tag}, {src_i_tag}) | no_sorg=({nosrc_v_tag}, {nosrc_i_tag})")
    print('Anteprima tabella input sorgente:')
    print(src_selected_df.head(10).to_string(index=False))
    print('Anteprima tabella input no_sorg:')
    print(nosrc_selected_df.head(10).to_string(index=False))
    print('Anteprima tabella finale interpolata (solo punti validi):')
    print(table_df.head(10).to_string(index=False))

    summary_rows.append({
        'miscela':                    mixture,
        'canale_tensione_sorg':        src_v_tag,
        'canale_corrente_sorg':        src_i_tag,
        'canale_tensione_no_sorg':     nosrc_v_tag,
        'canale_corrente_no_sorg':     nosrc_i_tag,
        'n_righe_originali':           int(n_total),
        'n_righe':                     len(table_df),
        'n_scartate_corrente_mancante': int(n_dropped_missing_current),
        'n_interp_valide':             int(n_valid),
        'n_fuori_range_tensione':      int(n_dropped_missing_current),
        'I2_no_sorg_abs_max':          float(table_df['I2_no_sorg_interp'].abs().max(skipna=True)),
        'root':                        str(root_path),
        'png':                         str(png_path),
        'csv':                         str(csv_path),
        'csv_input_sorg':              str(src_cols_csv),
        'csv_input_no_sorg':           str(nosrc_cols_csv),
    })

    try:
        canvas.Close()
    except Exception:
        pass

summary_df = pd.DataFrame(summary_rows)
print('\n===== RIEPILOGO =====')
print(summary_df.to_string(index=False))


[Ar_CO2_Iso_93_5_2] output_base = output/Ar_CO2_Iso_93_5_2/Ar_CO2_Iso_93_5_2_combined

=== Ar_CO2_Iso_93_5_2 ===
Colonne: sorgente=(V2, I2) | no_sorg=(V2, I2)
Anteprima tabella input sorgente:
                   time  V_V2_sorg  I_I2_sorg
2026-06-29 11:36:51.719      490.0   -0.00090
2026-06-29 11:36:52.599      490.0   -0.00090
2026-06-29 11:36:53.478      490.0   -0.00090
2026-06-29 11:36:54.356      490.0   -0.00095
2026-06-29 11:36:55.245      490.0   -0.00095
2026-06-29 11:36:56.123      490.0   -0.00090
2026-06-29 11:36:57.002      490.0   -0.00090
2026-06-29 11:36:57.879      490.0   -0.00090
2026-06-29 11:36:58.758      490.0   -0.00095
2026-06-29 11:36:59.647      490.0   -0.00095
Anteprima tabella input no_sorg:
                   time  V_V2_no_sorg  I_I2_no_sorg
2026-06-01 14:29:00.158         550.2       -0.0019
2026-06-01 14:29:01.037         550.2       -0.0019
2026-06-01 14:29:01.917         550.2       -0.0019
2026-06-01 14:29:02.786         550.2       -0.0019
2026-06-

Info in <TCanvas::Print>: png file output/Ar_CO2_Iso_93_5_2/Ar_CO2_Iso_93_5_2_combined.png has been created
Info in <TCanvas::Print>: png file output/Ar_CO2_Iso_88_10_2/Ar_CO2_Iso_88_10_2_combined.png has been created


In [47]:
# Cella legacy (single run basato su variabili globali V3/I3) disattivata
print("Questa cella legacy è stata disattivata per evitare conflitti con la logica per miscele (V2/I2).")
print("Usa la cella 6 per i plot I2_diff vs V2 per miscela.")

Questa cella legacy è stata disattivata per evitare conflitti con la logica per miscele (V2/I2).
Usa la cella 6 per i plot I2_diff vs V2 per miscela.


In [48]:
# Diagnostica corrente: con sorgente, senza sorgente e differenza (logica per miscela, V2/I2)
from pathlib import Path

if 'table_df' in globals() and {'time', 'I2_sorg', 'I2_no_sorg_interp', 'I2_diff'}.issubset(table_df.columns):
    check_df = table_df[['time', 'I2_sorg', 'I2_no_sorg_interp', 'I2_diff']].copy()
elif 'merged' in globals() and {'time', 'I2_sorg', 'I2_no_sorg_interp', 'I2_diff'}.issubset(merged.columns):
    check_df = merged[['time', 'I2_sorg', 'I2_no_sorg_interp', 'I2_diff']].copy()
else:
    raise RuntimeError("Variabili non disponibili: esegui prima la pipeline per miscele (cella 8).")

print('Prime 20 righe:')
print(check_df.head(20).to_string(index=False))

print('\nUltime 20 righe:')
print(check_df.tail(20).to_string(index=False))

print('\nStatistiche principali:')
print(check_df[['I2_sorg', 'I2_no_sorg_interp', 'I2_diff']].describe().to_string())

is_nosorg_all_zero = (check_df['I2_no_sorg_interp'].abs() < 1e-12).all()
print(f"\nI2_no_sorg_interp è (quasi) sempre zero? {is_nosorg_all_zero}")
print(f"Valore assoluto massimo I2_no_sorg_interp: {check_df['I2_no_sorg_interp'].abs().max():.6g}")

if 'mixture' in globals() and isinstance(mixture, str):
    csv_path = Path('output') / mixture / f'{mixture}_I_table_debug.csv'
else:
    csv_path = Path('output') / 'I_table_debug.csv'
csv_path.parent.mkdir(parents=True, exist_ok=True)

check_df.to_csv(csv_path, index=False)
print(f"CSV scritto in: {csv_path.resolve()}")

Prime 20 righe:
                   time  I2_sorg  I2_no_sorg_interp  I2_diff
2026-06-29 15:51:30.155 -0.00200                NaN      NaN
2026-06-29 15:51:31.401 -0.00225          -0.003108 0.000858
2026-06-29 15:51:32.291 -0.00230          -0.003108 0.000808
2026-06-29 15:51:33.171 -0.00230          -0.003108 0.000808
2026-06-29 15:51:34.051 -0.00230          -0.003108 0.000808
2026-06-29 15:51:34.929 -0.00230          -0.003108 0.000808
2026-06-29 15:51:35.810 -0.00240          -0.003108 0.000708
2026-06-29 15:51:36.690 -0.00240          -0.003108 0.000708
2026-06-29 15:51:37.559 -0.00240          -0.003108 0.000708
2026-06-29 15:51:38.439 -0.00240          -0.003108 0.000708
2026-06-29 15:51:39.318 -0.00240          -0.003108 0.000708
2026-06-29 15:51:40.564 -0.00240          -0.003108 0.000708
2026-06-29 15:51:41.454 -0.00240          -0.003108 0.000708
2026-06-29 15:51:42.400 -0.00240          -0.003108 0.000708
2026-06-29 15:51:43.280 -0.00240          -0.003108 0.000708
2026-06-

In [55]:
# Loader automatico gain_data per miscela: cerca in output/<miscela>/gain_data.py oppure .json
from pathlib import Path
import importlib.util
import json

base_dir = Path.cwd()
output_dir = base_dir / 'output'

mixture_configs = globals().get('MIXTURE_CONFIGS', [])
mixtures = [cfg.get('name') for cfg in mixture_configs if isinstance(cfg, dict) and cfg.get('name')]

if not mixtures and output_dir.exists():
    mixtures = [p.name for p in output_dir.iterdir() if p.is_dir()]

gain_data = {}

for mixture in mixtures:
    py_gain = output_dir / mixture / 'gain_data.py'
    json_gain = output_dir / mixture / 'gain_data.json'

    if py_gain.exists():
        spec = importlib.util.spec_from_file_location(f'gain_data_module_{mixture}', str(py_gain))
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        loaded = getattr(mod, 'gain_data', {})
        if mixture in loaded and isinstance(loaded[mixture], dict):
            gain_data[mixture] = {float(v): float(g) for v, g in loaded[mixture].items()}
        else:
            gain_data[mixture] = {float(v): float(g) for v, g in loaded.items()} if isinstance(loaded, dict) else {}
        print(f'[{mixture}] Caricato gain_data da file Python: {py_gain}')
    elif json_gain.exists():
        raw = json.loads(json_gain.read_text(encoding='utf-8'))
        if mixture in raw and isinstance(raw[mixture], dict):
            gain_data[mixture] = {float(v): float(g) for v, g in raw[mixture].items()}
        else:
            gain_data[mixture] = {float(v): float(g) for v, g in raw.items()} if isinstance(raw, dict) else {}
        print(f'[{mixture}] Caricato gain_data da file JSON: {json_gain}')
    else:
        gain_data[mixture] = {}
        print(f'[{mixture}] Nessun gain_data.py/json trovato: uso dizionario vuoto.')

print('Miscele disponibili in gain_data:', list(gain_data.keys()))

[Ar_CO2_Iso_93_5_2] Caricato gain_data da file Python: /Users/adelinadonofrio/Desktop/MPGD/output/Ar_CO2_Iso_93_5_2/gain_data.py
[Ar_CO2_Iso_88_10_2] Caricato gain_data da file Python: /Users/adelinadonofrio/Desktop/MPGD/output/Ar_CO2_Iso_88_10_2/gain_data.py
Miscele disponibili in gain_data: ['Ar_CO2_Iso_93_5_2', 'Ar_CO2_Iso_88_10_2']


In [56]:
import array
import numpy as np
import ROOT
import pandas as pd
from pathlib import Path

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)

if 'gain_data' not in globals() or gain_data is None:
    gain_data = {}
    print('Attenzione: gain_data non caricato dal loader, uso dizionario vuoto.')

output_dir = Path('output')
mixture_configs = globals().get('MIXTURE_CONFIGS', [])
mixtures = [cfg.get('name') for cfg in mixture_configs if isinstance(cfg, dict) and cfg.get('name')]
cfg_by_name = {cfg.get('name'): cfg for cfg in mixture_configs if isinstance(cfg, dict) and cfg.get('name')}
VOLTAGE_BIN_STEP = 5.0

if not mixtures and output_dir.exists():
    mixtures = [p.name for p in output_dir.iterdir() if p.is_dir()]

palette = [
    ROOT.TColor.GetColor('#D55E00'),
    ROOT.TColor.GetColor('#009E73'),
    ROOT.TColor.GetColor('#CC79A7'),
    ROOT.TColor.GetColor('#0072B2'),
    ROOT.TColor.GetColor('#E69F00'),
    ROOT.TColor.GetColor('#56B4E9'),
]

for idx, mixture in enumerate(mixtures):
    mixture_out = output_dir / mixture
    csv_candidates = [
        mixture_out / f'{mixture}_I_table.csv',
        mixture_out / f'{mixture}_I_table_debug.csv',
    ]
    csv_path = next((p for p in csv_candidates if p.exists()), None)
    if csv_path is None:
        print(f'[{mixture}] nessun CSV trovato, salto.')
        continue

    table_df = pd.read_csv(csv_path)
    v_candidates = [c for c in table_df.columns if c.startswith('V2')]
    if not v_candidates:
        v_candidates = [c for c in table_df.columns if c.startswith('V_')]
    if not v_candidates:
        print(f'[{mixture}] colonna tensione non trovata, salto.')
        continue
    v_col = v_candidates[0]

    diff_col = 'I2_diff' if 'I2_diff' in table_df.columns else ('I_diff' if 'I_diff' in table_df.columns else 'I3_diff')
    if diff_col not in table_df.columns:
        print(f'[{mixture}] colonna differenza non trovata, salto.')
        continue

    work = table_df[[v_col, diff_col]].copy()
    work.columns = ['V', 'I_diff']
    # Allineamento con il plot I2_diff: binning a step di 5 V
    work['V_round'] = (work['V'] / VOLTAGE_BIN_STEP).round() * VOLTAGE_BIN_STEP
    grouped = work.groupby('V_round')['I_diff'].agg(['mean', 'std', 'count']).reset_index()
    grouped.columns = ['V', 'I_diff_mean', 'I_diff_std', 'n']
    grouped = grouped[grouped['I_diff_mean'] > 0].sort_values('V').reset_index(drop=True)

    if grouped.empty:
        print(f'[{mixture}] nessun punto positivo per I_diff, salto.')
        continue

    color_i = palette[idx % len(palette)]
    color_g = palette[(idx + 2) % len(palette)]

    v_i_arr  = array.array('d', grouped['V'].to_numpy(dtype=float))
    i_arr    = array.array('d', grouped['I_diff_mean'].to_numpy(dtype=float))
    ev_i_arr = array.array('d', np.zeros(len(grouped), dtype=float))
    ei_arr   = array.array('d', (grouped['I_diff_std'] / np.sqrt(grouped['n'])).fillna(0).to_numpy(dtype=float))

    g_current = ROOT.TGraphErrors(len(v_i_arr), v_i_arr, i_arr, ev_i_arr, ei_arr)
    g_current.SetTitle('')
    g_current.SetMarkerStyle(20)
    g_current.SetMarkerSize(0.8)
    g_current.SetMarkerColor(color_i)
    g_current.SetLineColor(color_i)
    g_current.SetLineWidth(2)

    gd_raw = gain_data.get(mixture, {}) if isinstance(gain_data, dict) else {}
    gain_min_v = cfg_by_name.get(mixture, {}).get('gain_peak_min_voltage', None)
    if gain_min_v is not None:
        gd = {float(v): float(g) for v, g in gd_raw.items() if float(v) >= float(gain_min_v)}
        print(f'[{mixture}] guadagno: applicato taglio V >= {float(gain_min_v):g} ({len(gd)}/{len(gd_raw)} punti)')
    else:
        gd = gd_raw

    has_gain = len(gd) > 0

    c_cg = ROOT.TCanvas(f'c_{mixture}_I_gain', f'Corrente e Guadagno - {mixture}', 1000, 700)
    c_cg.SetGrid()
    c_cg.SetLeftMargin(0.13)
    c_cg.SetRightMargin(0.16)

    g_current.Draw('APL')
    g_current.GetXaxis().SetTitle('Tensione [V]')
    g_current.GetYaxis().SetTitle('I_{2,diff} media [#muA]')
    g_current.GetYaxis().SetTitleColor(color_i)
    g_current.GetYaxis().SetLabelColor(color_i)
    g_current.GetYaxis().SetAxisColor(color_i)
    c_cg.SetLogy(0)

    # Legge il range reale visualizzato sull'asse sinistro
    c_cg.Modified()
    c_cg.Update()
    y_left_min = float(g_current.GetYaxis().GetXmin())
    y_left_max = float(g_current.GetYaxis().GetXmax())

    leg_cg = ROOT.TLegend(0.15, 0.75, 0.55, 0.90)
    leg_cg.SetFillStyle(0)
    leg_cg.SetBorderSize(0)
    leg_cg.SetTextSize(0.035)
    leg_cg.AddEntry(g_current, 'I_{2,diff} = I_{2,sorg} - I_{2,no sorg}', 'lp')

    if has_gain:
        gain_min = float(min(gd.values()))
        gain_max = float(max(gd.values()))

        if (gain_max - gain_min) > 0 and (y_left_max - y_left_min) > 0:
            scale_g = (y_left_max - y_left_min) / (gain_max - gain_min)
            offset_g = y_left_min - gain_min * scale_g
        else:
            scale_g = 1.0
            offset_g = 0.0

        v_gain_scaled = array.array('d', sorted(gd.keys()))
        gain_scaled = array.array('d', [gd[v] * scale_g + offset_g for v in sorted(gd.keys())])
        g_gain_scaled = ROOT.TGraph(len(v_gain_scaled), v_gain_scaled, gain_scaled)
        g_gain_scaled.SetMarkerStyle(21)
        g_gain_scaled.SetMarkerSize(0.8)
        g_gain_scaled.SetMarkerColor(color_g)
        g_gain_scaled.SetLineColor(color_g)
        g_gain_scaled.SetLineWidth(2)
        g_gain_scaled.Draw('PL same')

        # Asse destro rosa ancorato al bordo destro del pad e coerente col range mostrato
        x_right = ROOT.gPad.GetUxmax()
        axis_gain = ROOT.TGaxis(x_right, y_left_min, x_right, y_left_max, gain_min, gain_max, 510, '+R')
        axis_gain.SetTitle('Guadagno (picco)')
        axis_gain.SetTitleColor(color_g)
        axis_gain.SetLabelColor(color_g)
        axis_gain.SetLineColor(color_g)
        axis_gain.Draw()

        leg_cg.AddEntry(g_gain_scaled, 'Guadagno (posizione picco)', 'lp')

    leg_cg.Draw()
    c_cg.Modified()
    c_cg.Update()

    out_base = mixture_out / f'{mixture}_I_vs_Gain_vs_V'
    c_cg.SaveAs(str(out_base.with_suffix('.root')))
    c_cg.SaveAs(str(out_base.with_suffix('.png')))
    print(f'[{mixture}] Salvato: {out_base}.root / .png')

[Ar_CO2_Iso_93_5_2] Salvato: output/Ar_CO2_Iso_93_5_2/Ar_CO2_Iso_93_5_2_I_vs_Gain_vs_V.root / .png
[Ar_CO2_Iso_88_10_2] guadagno: applicato taglio V >= 530 (8/12 punti)
[Ar_CO2_Iso_88_10_2] Salvato: output/Ar_CO2_Iso_88_10_2/Ar_CO2_Iso_88_10_2_I_vs_Gain_vs_V.root / .png


Info in <TCanvas::SaveAs>: ROOT file output/Ar_CO2_Iso_93_5_2/Ar_CO2_Iso_93_5_2_I_vs_Gain_vs_V.root has been created
Info in <TCanvas::Print>: png file output/Ar_CO2_Iso_93_5_2/Ar_CO2_Iso_93_5_2_I_vs_Gain_vs_V.png has been created
Info in <TCanvas::SaveAs>: ROOT file output/Ar_CO2_Iso_88_10_2/Ar_CO2_Iso_88_10_2_I_vs_Gain_vs_V.root has been created
Info in <TCanvas::Print>: png file output/Ar_CO2_Iso_88_10_2/Ar_CO2_Iso_88_10_2_I_vs_Gain_vs_V.png has been created
